# Unlimited-OCR — GPU server for the AI OCR Field Test

Baidu **Unlimited-OCR** (`baidu/Unlimited-OCR`) is GPU-only, so it cannot run on the (CPU) machine where the harness lives. This notebook runs it on a Colab GPU and exposes its **OpenAI-compatible SGLang server** through a public tunnel. You then paste that URL into your local `.env` and run the harness normally with `--models unlimited_ocr`.

**GPU notes**
- Set the runtime to a **GPU** (Runtime -> Change runtime type -> GPU).
- The repo's default `--attention-backend fa3` needs an **H100**. This notebook uses `flashinfer`, which works on **T4 / L4 / A100**.
- Free **T4 (15 GB)** may work for single-page docs but can OOM on the 10-page PDF / 32K context. For reliability use **Colab Pro (L4 / A100)**.

Keep this tab running while the local harness calls the endpoint.

In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 2) Install Unlimited-OCR + SGLang (uses the repo's bundled SGLang wheel)
!git clone https://github.com/baidu/Unlimited-OCR.git
%cd Unlimited-OCR
# Bundled dev wheel ships in the repo's wheel/ folder; glob in case the name changes.
!pip install -q wheel/sglang-*.whl
!pip install -q kernels==0.11.7 pymupdf==1.27.2.2 flashinfer-python huggingface_hub
# The model's trust_remote_code modeling file needs these extra packages:
!pip install -q addict einops easydict matplotlib psutil
print('install done')

In [ ]:
# 3) Launch the SGLang OpenAI-compatible server in the background.
#    flashinfer backend (T4/L4/A100). Memory tuned to survive on a free T4 (15 GB):
#    mem-fraction 0.7 + context 8192. On L4/A100 you can raise these (0.85 / 32768).
import subprocess, time, requests

log = open('sglang.log', 'w')
server = subprocess.Popen([
    'python', '-m', 'sglang.launch_server',
    '--model', 'baidu/Unlimited-OCR',
    '--served-model-name', 'Unlimited-OCR',
    '--attention-backend', 'flashinfer',
    '--page-size', '1',
    '--mem-fraction-static', '0.7',
    '--context-length', '8192',
    '--max-running-requests', '1',
    '--enable-custom-logit-processor',
    '--disable-overlap-schedule',
    '--trust-remote-code',
    '--host', '0.0.0.0', '--port', '10000',
], stdout=log, stderr=subprocess.STDOUT)

print('Loading model / starting server (can take several minutes on first run)...')
ready = False
for i in range(180):
    try:
        if requests.get('http://127.0.0.1:10000/v1/models', timeout=2).status_code == 200:
            ready = True; print('\nServer is READY'); break
    except Exception:
        pass
    if server.poll() is not None:
        print('\nServer process exited early - see sglang.log below'); break
    time.sleep(5); print('.', end='')
if not ready:
    print('\nNot ready yet; inspect the log:')
!tail -n 30 sglang.log

In [ ]:
# 4) Open a free public tunnel to port 10000 with cloudflared, and print the URL.
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:10000'],
                          stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    try:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cf.log').read())
        if m:
            url = m.group(0); break
    except Exception:
        pass
print('\n' + '=' * 60)
if url:
    print('Add this line to your local ai-ocr-field-test/.env :\n')
    print('UNLIMITED_OCR_BASE_URL=' + url + '/v1')
else:
    print('Tunnel URL not found yet; check cf.log')
print('=' * 60)

In [ ]:
# 5) (Optional) Sanity-check the endpoint from inside Colab before using the harness.
import base64, io
from openai import OpenAI
from PIL import Image, ImageDraw
client = OpenAI(base_url='http://127.0.0.1:10000/v1', api_key='EMPTY')
img = Image.new('RGB', (400, 80), 'white')
ImageDraw.Draw(img).text((10, 30), 'Invoice Total: 96.47', fill='black')
buf = io.BytesIO(); img.save(buf, format='PNG')
b64 = base64.b64encode(buf.getvalue()).decode()
r = client.chat.completions.create(model='Unlimited-OCR', messages=[{'role': 'user', 'content': [
    {'type': 'text', 'text': 'document parsing.'},
    {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,' + b64}}]}])
print(r.choices[0].message.content)

## Then, on your local machine
1. Paste the printed `UNLIMITED_OCR_BASE_URL=...` line into `ai-ocr-field-test/.env`.
2. Run the harness against the live endpoint:
   ```
   venv\Scripts\python scripts\run_experiment.py --models unlimited_ocr
   venv\Scripts\python scripts\init_metrics.py
   venv\Scripts\python scripts\generate_report.py
   ```
3. Keep this Colab tab open while the run is in progress (the tunnel must stay up).

Outputs land in `outputs/unlimited_ocr/{test_id}_output.md`, just like the other models.